In [117]:
import pandas as pd
Crick_H3N2 = pd.read_excel('/mnt/chenyihao/workspace/fluProfiler_source/data/raw/data4model(Crick-H3N2).xlsx')
Crick_H3N2_serum = Crick_H3N2[['serumName', 'serumPassage', 'serumPassCat', 'serumDate', 'serumType',
                               'serumIslID', 'serumMatchedPass', 'serumHA', 'serumNA']].copy()
Crick_H3N2_virus = Crick_H3N2[['virusName', 'virusPassage', 'virusPassCat', 'virusDate', 'virusType',
                               'virusIslID', 'virusMatchedPass', 'virusHA', 'virusNA']].copy()
Crick_H3N2_serum.columns = Crick_H3N2_virus.columns
Crick_H3N2_serum['virusType'] = 'Serum'
Crick_H3N2_virus['virusType'] = 'Virus'
Crick_H3N2_all_strain = pd.concat([Crick_H3N2_serum, Crick_H3N2_virus], axis=0)

In [21]:
def read_fasta(file_path):
    sequences = {}
    with open(file_path, 'r') as file:
        lines = file.readlines()
        current_id = None
        current_sequence = []
        
        for line in lines:
            line = line.strip()
            if line.startswith('>'):  # 新序列的开始
                if current_id is not None:
                    sequences[current_id] = ''.join(current_sequence)
                current_id = line[1:]  # 去掉 '>'
                current_sequence = []
            else:
                current_sequence.append(line)
        
        # 添加最后一个序列
        if current_id is not None:
            sequences[current_id] = ''.join(current_sequence)
    
    return sequences

# fasta_file = "/mnt/chenyihao/workspace/fluProfiler_source/data/raw/H3N2_DNA.fasta"
# H3N2_DNA_dict = read_fasta(fasta_file)

In [ ]:
# virusName = []
# EPI_ISL_ID = []
# Passage = []
# clade = []
# collectionDate = []
# EPI_ID = []
# Sequence = []
# for key, value in H3N2_DNA_dict.items():
#     temp = key.split('|')
#     virusName.append(temp[0])
#     EPI_ISL_ID.append(temp[1])
#     Passage.append(temp[2])
#     clade.append(temp[3])
#     collectionDate.append(temp[4])
#     EPI_ID.append(temp[5])
    
#     Sequence.append(value)

# H3N2_DNA_data = pd.DataFrame({'virusName': virusName, 'EPI_ISL_ID': EPI_ISL_ID, 'Passage': Passage, 
#                               'clade': clade, 'collectionDate': collectionDate, 'EPI_ID': EPI_ID, 
#                               'Sequence': Sequence})

In [196]:
start_date = '2012-01-01'
end_date = '2018-12-31'
## 筛选指定日期内的
strain_filt1 = Crick_H3N2_all_strain[(Crick_H3N2_all_strain['virusDate'] >= start_date) &
                                     (Crick_H3N2_all_strain['virusDate'] <= end_date)]
print(len(strain_filt1))
## 去除virusIslID重复的行
strain_filt2 = strain_filt1.drop_duplicates(subset=['virusIslID'])
print(len(strain_filt2))
## 去除鸡胚株
strain_filt3 = strain_filt2[strain_filt2['virusPassCat'] == 'CELL'].reset_index(drop=True)
print(len(strain_filt3))

61295
2306
2212


In [199]:
DNA = pd.read_csv('Crick_H3N2_DNA.csv')
## 保留合适日期
DNA_filt1 = DNA[(DNA['collectionDate'] >= start_date) &
                (DNA['collectionDate'] <= end_date)]
## 保留细胞株
DNA_filt2 = DNA_filt1[DNA_filt1['PassCat'] == 'CELL'].reset_index(drop=True)

In [216]:
def build_AA_DF(AA_dict):
    result = []
    for key, value in AA_dict.items():
        temp = key.split('|')[0:-1]
        if(len(temp) != 15):
            continue
        result.append(temp + [value])
    return result
H3_AA_dict = read_fasta('/mnt/chenyihao/workspace/fluProfiler_source/data/raw/H3_AA.fasta')
N2_AA_dict = read_fasta('/mnt/chenyihao/workspace/fluProfiler_source/data/raw/N2_AA.fasta')

In [249]:
# H3_AA_DF = pd.DataFrame(build_AA_DF(H3_AA_dict)).iloc[:,[0,1,6,14,15]]
# N2_AA_DF = pd.DataFrame(build_AA_DF(N2_AA_dict)).iloc[:,[0,1,6,14,15]]
# H3_AA_DF.columns = ['IsolateName','EPI_ISL_ID','CollectionDate','EPI_ID','HA_seq']
# N2_AA_DF.columns = ['IsolateName','EPI_ISL_ID','CollectionDate','EPI_ID','Sequence']
H3_AA_DF = pd.DataFrame(build_AA_DF(H3_AA_dict)).iloc[:,[6,14,15]]
N2_AA_DF = pd.DataFrame(build_AA_DF(N2_AA_dict)).iloc[:,[6,1,15]]
H3_AA_DF.columns = ['CollectionDate','EPI_ID','HA_seq']
N2_AA_DF.columns = ['CollectionDate','EPI_ISL_ID','NA_seq']
H3_AA_DF['CollectionDate'] = pd.to_datetime(H3_AA_DF['CollectionDate'],format='%Y-%m-%d',errors='coerce')
N2_AA_DF['CollectionDate'] = pd.to_datetime(N2_AA_DF['CollectionDate'],format='%Y-%m-%d',errors='coerce')

In [250]:
## 筛选指定日期区间内的
H3_AA_DF_filt1 = H3_AA_DF[(H3_AA_DF['CollectionDate'] >= '2012-01-01') &
                          (H3_AA_DF['CollectionDate'] <= '2021-12-31')].reset_index(drop=True)
N2_AA_DF_filt1 = N2_AA_DF[(N2_AA_DF['CollectionDate'] >= '2012-01-01') &
                          (N2_AA_DF['CollectionDate'] <= '2021-12-31')].reset_index(drop=True)
print('H3',len(H3_AA_DF_filt1))
print('N2',len(N2_AA_DF_filt1))

H3 94311
N2 77687


In [256]:
merge1 = pd.merge(DNA_filt2,H3_AA_DF_filt1,on='EPI_ID',how='left').drop(columns='CollectionDate')
merge2 = pd.merge(merge1,N2_AA_DF_filt1,on='EPI_ISL_ID',how='left').drop(columns='CollectionDate')

In [269]:
H3N2_meta = pd.read_csv('Meta_combined.csv', low_memory=False)[['Isolate_Id','Location']]
H3N2_meta.columns = ['EPI_ISL_ID','Location']

In [270]:
merge3 = pd.merge(merge2,H3N2_meta,on='EPI_ISL_ID',how='left')

In [278]:
merge4 = merge3.drop_duplicates(subset=['virusName','Location','collectionDate'])

In [334]:
## 保留HA蛋白长566、NA蛋白长469的且没有na信息的行
merge5 = merge4[(merge4['HA_seq'].str.len() == 566) & 
       (merge4['NA_seq'].str.len() == 469)].dropna().reset_index(drop=True)
## 把行按照日期排列，去除Sequence重复的行，保留日期最早的
merge6 = merge5.sort_values(by='collectionDate').drop_duplicates(subset=['Sequence']).reset_index(drop=True)

In [340]:
Location = []
for str in merge6['Location']:
    temp = str.split('/')
    Location.append(temp[0] + '/' + temp[1])

In [345]:
merge7 = merge6.copy()
merge7['Location'] = Location

In [348]:
merge7

,virusName,EPI_ISL_ID,Passage,clade,collectionDate,EPI_ID,Sequence,PassCat,HA_seq,NA_seq,Location
0,A/Stockholm/12-16700/2012,EPI_ISL_134178,MDCK1/SIAT1,3C.2,2012-01-01,EPI416524,atgaagactatcattgctttgagctacattctatgtctggttttcg...,CELL,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / Sweden
1,A/Praha/130/2012,EPI_ISL_121953,MDCK3/SIAT2,3C.2,2012-01-01,EPI377379,atgaagactatcattgctttgagctacattctatgtctggttttcg...,CELL,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / Czech Republic
2,A/Austria/653679/2012,EPI_ISL_107832,C1/SIAT1,3C.2,2012-01-01,EPI358903,atgaagactatcatagctttgagctacattctatgtctggttttcg...,CELL,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / Austria
3,A/Baden-Wurttemberg/2/2012,EPI_ISL_104125,C2/SIAT1,3C.2,2012-01-01,EPI354166,atgaagactatcattgcttttgactacattctacgtctggttttcg...,CELL,MKTIIAFDYILRLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / Germany
4,A/Tehran/104/2012,EPI_ISL_103312,MDCK1/SIAT1,3C.2,2012-01-01,EPI352815,atgaagactatcattgctttgagctacattctatgtctggttttcg...,CELL,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,"Asia / Iran, Islamic Republic of"
...,...,...,...,...,...,...,...,...,...,...,...
1782,A/Ukraine/8106/2018,EPI_ISL_342038,MDCK2/SIAT1,3C.2a1b.1,2018-12-27,EPI1371579,atgaagactatcattgctttgagctacattctatgtctggttttcg...,CELL,MKTIIALSYILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / Ukraine
1783,A/Lyon/2335/2018,EPI_ISL_355066,MDCK2/SIAT1,3C.3a1,2018-12-27,EPI1436501,atgaagactatcattgctttgagctgcattctatgtctggttttcg...,CELL,MKTIIALSCILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / France
1784,A/Valladolid/560/2018,EPI_ISL_355128,SIAT1/SIAT1,3C.3a1,2018-12-27,EPI1436563,atgaagactatcattgctttgagctgcattctatgtctggttttcg...,CELL,MKTIIALSCILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / Spain
1785,A/Komarno/58/2018,EPI_ISL_377319,MDCK1,3C.2a1b.1,2018-12-28,EPI1543130,atgaagactatcattgctttgagctacattctatgtctggttttcg...,CELL,MKTIIALSYILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MNPNQKIITIGSVSLTISTICFFMQIAILITTVTLHFKQYEFNSPP...,Europe / Slovakia


In [347]:
input_file = 'input.fasta'

# 打开文件并写入
with open(input_file, 'w') as fasta_file:
    for index, row in merge6.iterrows():
        # 写入序列名字
        fasta_file.write(f">{row['virusName']}\n")
        # 写入序列
        fasta_file.write(f"{row['Sequence']}\n")